# Filling missing values in pandas

An offline practice notebook for Python Lesson 15. Install pandas, NumPy and scikit-learn in the notebook's Python environment.

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "A": [1.0, np.nan, 3.0, 1.0, np.nan],
    "B": [np.nan, 10.0, 20.0, np.nan, 20.0],
})
print(df)
print(df.isna().sum())

## Directional fills and constants

Forward and backward fill depend on row order. For prediction, backward fill can use future observations and cause leakage.

In [ ]:
print(df.ffill())
print(df.bfill())
print(df.fillna({"A": 0, "B": -1}))
assert pd.isna(df.ffill().iloc[0, 1])
assert pd.isna(df.bfill().iloc[-1, 0])

## Mean, median, mode and sampling

Pick one statistic based on the data and the problem. A column with multiple modes needs an explicit tie rule. Learn fill values from the training set when building a predictive model.

In [ ]:
print(df.fillna(df.mean()))
print(df.fillna(df.median()))
print(df.fillna(df.mode().iloc[0]))
rng = np.random.default_rng(42)
sampled = df.copy()
for column in sampled:
    missing = sampled[column].isna()
    observed = sampled.loc[~missing, column]
    if not observed.empty:
        sampled.loc[missing, column] = rng.choice(observed, missing.sum(), replace=True)
print(sampled)

## scikit-learn imputation

The toy data below serves as training data. Fit on training data and transform held-out rows with the fitted imputer. MissingIndicator supplies Boolean features. IterativeImputer requires the documented experimental import.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer, MissingIndicator

holdout = pd.DataFrame({"A": [np.nan], "B": [15.0]})
for imputer in (SimpleImputer(strategy="median"), KNNImputer(n_neighbors=2),
                IterativeImputer(random_state=42)):
    print(type(imputer).__name__, imputer.fit(df).transform(holdout))
print(MissingIndicator(features="all").fit(df).transform(holdout))

## Practise dataset inspection

This offline example uses illustrative rows. In class, the instructor compared a local Titanic CSV with Seaborn's dataset. Check each version's shape and columns before comparing outputs.

In [ ]:
passengers = pd.DataFrame({
    "age": [22.0, 38.0, np.nan, 35.0],
    "fare": [7.25, 71.28, 7.92, 53.10],
    "embarked": ["S", "C", "S", np.nan],
})
print(passengers.describe())
print(passengers.isna().sum())
print(passengers.loc[:, ["age", "fare"]].iloc[::2])
for index, row in passengers.iterrows():
    print(index, row["fare"])